# Lesson 11b: Model-Based Reinforcement Learning Practical

**Learning Objectives:**
- Implement Dyna-Q for tabular environments
- Build neural network dynamics models with ensembles
- Implement Model Predictive Control (MPC) with CEM planning
- Create MBPO-style model-based policy optimization
- Compare sample efficiency of model-based vs model-free
- Visualize model predictions and uncertainty

**Prerequisites:** Q-Learning (4b), DQN (7b), SAC (10b), Model-Based Theory (11a)

**Environment:** Google Colab with GPU support recommended

In [ ]:
# Install dependencies
!pip install gymnasium numpy matplotlib torch tqdm -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import defaultdict, deque
import random
import gymnasium as gym
from tqdm import tqdm
import copy
from typing import List, Tuple

# Set random seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dyna-Q Implementation

In [ ]:
class DynaQAgent:
    """Dyna-Q: Integrates learning and planning with tabular model."""
    
    def __init__(self, n_states: int, n_actions: int,
                 alpha: float = 0.1, gamma: float = 0.95,
                 epsilon: float = 0.1, n_planning_steps: int = 5):
        """
        Args:
            n_states: Number of states
            n_actions: Number of actions
            alpha: Learning rate
            gamma: Discount factor
            epsilon: Exploration rate
            n_planning_steps: Number of planning updates per real step
        """
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_planning_steps = n_planning_steps
        
        # Q-table
        self.Q = np.zeros((n_states, n_actions))
        
        # Model: stores (reward, next_state) for each (state, action)
        self.model = {}  # {(s, a): (r, s')}
        
        # Track visited state-action pairs
        self.visited = set()
    
    def select_action(self, state: int) -> int:
        """Epsilon-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])
    
    def update(self, state: int, action: int, reward: float, next_state: int):
        """Single Dyna-Q update step."""
        # (a) Direct RL: Update Q-value from real experience
        td_target = reward + self.gamma * np.max(self.Q[next_state])
        td_error = td_target - self.Q[state, action]
        self.Q[state, action] += self.alpha * td_error
        
        # (c) Model learning: Store transition in model
        self.model[(state, action)] = (reward, next_state)
        self.visited.add((state, action))
        
        # (d) Planning: Simulated experience
        for _ in range(self.n_planning_steps):
            if not self.visited:
                break
            
            # Sample random previously visited (s, a)
            s, a = random.choice(list(self.visited))
            
            # Get simulated experience from model
            r, s_next = self.model[(s, a)]
            
            # Update Q-value from simulated experience
            td_target = r + self.gamma * np.max(self.Q[s_next])
            td_error = td_target - self.Q[s, a]
            self.Q[s, a] += self.alpha * td_error


# Test on a simple gridworld
env = gym.make('FrozenLake-v1', is_slippery=False)
agent = DynaQAgent(
    n_states=env.observation_space.n,
    n_actions=env.action_space.n,
    n_planning_steps=10
)

print(f"Environment: FrozenLake")
print(f"States: {env.observation_space.n}")
print(f"Actions: {env.action_space.n}")
print(f"Dyna-Q agent created with {agent.n_planning_steps} planning steps")

In [ ]:
def train_dyna_q(agent, env, n_episodes: int = 500):
    """Train Dyna-Q agent."""
    episode_rewards = []
    
    for episode in tqdm(range(n_episodes)):
        state, _ = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.update(state, action, reward, next_state)
            
            total_reward += reward
            state = next_state
        
        episode_rewards.append(total_reward)
    
    return episode_rewards


# Train Dyna-Q
print("Training Dyna-Q...")
dyna_rewards = train_dyna_q(agent, env, n_episodes=500)

# Compare with vanilla Q-learning (n_planning_steps=0)
print("\nTraining vanilla Q-learning...")
q_agent = DynaQAgent(
    n_states=env.observation_space.n,
    n_actions=env.action_space.n,
    n_planning_steps=0  # No planning
)
q_rewards = train_dyna_q(q_agent, env, n_episodes=500)

# Plot comparison
plt.figure(figsize=(10, 5))
window = 50
plt.plot(np.convolve(dyna_rewards, np.ones(window)/window, mode='valid'),
         label=f'Dyna-Q (planning={agent.n_planning_steps})', alpha=0.8)
plt.plot(np.convolve(q_rewards, np.ones(window)/window, mode='valid'),
         label='Q-Learning (planning=0)', alpha=0.8)
plt.xlabel('Episode')
plt.ylabel('Return (smoothed)')
plt.title('Dyna-Q vs Q-Learning on FrozenLake')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/tmp/dyna_q_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDyna-Q final performance: {np.mean(dyna_rewards[-50:]):.3f}")
print(f"Q-Learning final performance: {np.mean(q_rewards[-50:]):.3f}")

## 2. Neural Network Dynamics Model

In [ ]:
class DynamicsModel(nn.Module):
    """Neural network dynamics model: (s, a) → (s', r)."""
    
    def __init__(self, state_dim: int, action_dim: int,
                 hidden_sizes: List[int] = [256, 256]):
        super(DynamicsModel, self).__init__()
        
        # Build network
        layers = []
        input_dim = state_dim + action_dim
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_dim, hidden_size))
            layers.append(nn.ReLU())
            input_dim = hidden_size
        
        self.network = nn.Sequential(*layers)
        
        # Output heads: Δs (state change) and reward
        self.delta_head = nn.Linear(input_dim, state_dim)
        self.reward_head = nn.Linear(input_dim, 1)
    
    def forward(self, state, action):
        """Predict next state and reward."""
        x = torch.cat([state, action], dim=-1)
        features = self.network(x)
        
        # Predict delta (more stable than direct next state)
        delta = self.delta_head(features)
        next_state = state + delta
        
        reward = self.reward_head(features)
        
        return next_state, reward


class ProbabilisticDynamicsModel(nn.Module):
    """Probabilistic dynamics model: (s, a) → N(μ, σ²)."""
    
    def __init__(self, state_dim: int, action_dim: int,
                 hidden_sizes: List[int] = [256, 256],
                 min_log_std: float = -10, max_log_std: float = 0.5):
        super(ProbabilisticDynamicsModel, self).__init__()
        
        self.min_log_std = min_log_std
        self.max_log_std = max_log_std
        
        # Build network
        layers = []
        input_dim = state_dim + action_dim
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_dim, hidden_size))
            layers.append(nn.ReLU())
            input_dim = hidden_size
        
        self.network = nn.Sequential(*layers)
        
        # Output heads
        self.mean_head = nn.Linear(input_dim, state_dim)
        self.log_std_head = nn.Linear(input_dim, state_dim)
        self.reward_head = nn.Linear(input_dim, 1)
    
    def forward(self, state, action):
        """Predict distribution over next states and reward."""
        x = torch.cat([state, action], dim=-1)
        features = self.network(x)
        
        # Mean and std for next state delta
        delta_mean = self.mean_head(features)
        delta_log_std = self.log_std_head(features)
        delta_log_std = torch.clamp(delta_log_std, self.min_log_std, self.max_log_std)
        
        next_state_mean = state + delta_mean
        next_state_std = torch.exp(delta_log_std)
        
        reward = self.reward_head(features)
        
        return next_state_mean, next_state_std, reward
    
    def sample(self, state, action):
        """Sample next state."""
        mean, std, reward = self.forward(state, action)
        next_state = mean + std * torch.randn_like(mean)
        return next_state, reward


# Test models
state_dim, action_dim = 4, 2
det_model = DynamicsModel(state_dim, action_dim).to(device)
prob_model = ProbabilisticDynamicsModel(state_dim, action_dim).to(device)

test_state = torch.randn(10, state_dim).to(device)
test_action = torch.randn(10, action_dim).to(device)

# Deterministic model
next_state, reward = det_model(test_state, test_action)
print(f"Deterministic model output shapes:")
print(f"  Next state: {next_state.shape}")
print(f"  Reward: {reward.shape}")

# Probabilistic model
next_state_mean, next_state_std, reward = prob_model(test_state, test_action)
print(f"\nProbabilistic model output shapes:")
print(f"  Next state mean: {next_state_mean.shape}")
print(f"  Next state std: {next_state_std.shape}")
print(f"  Reward: {reward.shape}")

## 3. Ensemble Dynamics Model

In [ ]:
class EnsembleDynamicsModel:
    """Ensemble of probabilistic dynamics models for uncertainty estimation."""
    
    def __init__(self, state_dim: int, action_dim: int, n_models: int = 5,
                 hidden_sizes: List[int] = [256, 256], lr: float = 1e-3):
        """
        Args:
            n_models: Number of models in ensemble
        """
        self.n_models = n_models
        
        # Create ensemble
        self.models = [
            ProbabilisticDynamicsModel(state_dim, action_dim, hidden_sizes).to(device)
            for _ in range(n_models)
        ]
        
        # Separate optimizers
        self.optimizers = [
            optim.Adam(model.parameters(), lr=lr)
            for model in self.models
        ]
    
    def train_step(self, states, actions, next_states, rewards):
        """Train all models on batch with bootstrap sampling."""
        losses = []
        
        for model, optimizer in zip(self.models, self.optimizers):
            # Bootstrap sampling: sample with replacement
            indices = np.random.choice(len(states), len(states), replace=True)
            batch_states = states[indices]
            batch_actions = actions[indices]
            batch_next_states = next_states[indices]
            batch_rewards = rewards[indices]
            
            # Forward pass
            pred_mean, pred_std, pred_reward = model(batch_states, batch_actions)
            
            # Compute negative log-likelihood loss
            delta_true = batch_next_states - batch_states
            delta_pred = pred_mean - batch_states
            
            # Gaussian NLL
            state_loss = 0.5 * ((delta_true - delta_pred) / (pred_std + 1e-8))**2 + torch.log(pred_std + 1e-8)
            state_loss = state_loss.mean()
            
            reward_loss = F.mse_loss(pred_reward, batch_rewards)
            
            total_loss = state_loss + reward_loss
            
            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            losses.append(total_loss.item())
        
        return np.mean(losses)
    
    def predict(self, state, action, use_mean: bool = False):
        """Predict using ensemble.
        
        Args:
            use_mean: If True, return ensemble mean. If False, sample random model.
        """
        if use_mean:
            # Average predictions across ensemble
            predictions = [model(state, action) for model in self.models]
            next_state_mean = torch.stack([p[0] for p in predictions]).mean(dim=0)
            reward = torch.stack([p[2] for p in predictions]).mean(dim=0)
            return next_state_mean, reward
        else:
            # Sample random model (for planning)
            model = random.choice(self.models)
            return model.sample(state, action)
    
    def get_uncertainty(self, state, action):
        """Get ensemble disagreement as uncertainty measure."""
        predictions = [model(state, action)[0] for model in self.models]
        predictions = torch.stack(predictions)
        uncertainty = predictions.std(dim=0).mean(dim=-1)
        return uncertainty


# Test ensemble
ensemble = EnsembleDynamicsModel(state_dim=4, action_dim=2, n_models=5)
print(f"Ensemble with {ensemble.n_models} models created")

# Test prediction
test_state = torch.randn(5, 4).to(device)
test_action = torch.randn(5, 2).to(device)
next_state, reward = ensemble.predict(test_state, test_action, use_mean=True)
uncertainty = ensemble.get_uncertainty(test_state, test_action)

print(f"\nEnsemble prediction shapes:")
print(f"  Next state: {next_state.shape}")
print(f"  Uncertainty: {uncertainty.shape}")
print(f"  Mean uncertainty: {uncertainty.mean().item():.4f}")

## 4. Model Predictive Control (MPC) with CEM

In [ ]:
class CEMPlanner:
    """Cross-Entropy Method for planning with learned model."""
    
    def __init__(self, dynamics_model, action_dim: int, horizon: int = 10,
                 n_iterations: int = 5, population_size: int = 200,
                 elite_frac: float = 0.1, action_low: float = -1.0,
                 action_high: float = 1.0):
        """
        Args:
            dynamics_model: Learned dynamics model
            action_dim: Dimension of action space
            horizon: Planning horizon
            n_iterations: CEM iterations
            population_size: Number of action sequences to sample
            elite_frac: Fraction of top sequences to keep
            action_low/high: Action bounds
        """
        self.dynamics_model = dynamics_model
        self.action_dim = action_dim
        self.horizon = horizon
        self.n_iterations = n_iterations
        self.population_size = population_size
        self.n_elite = max(1, int(population_size * elite_frac))
        self.action_low = action_low
        self.action_high = action_high
    
    def plan(self, initial_state):
        """Plan action sequence using CEM.
        
        Returns:
            First action of optimized sequence
        """
        # Initialize distribution: mean and std for each action in sequence
        mean = np.zeros((self.horizon, self.action_dim))
        std = np.ones((self.horizon, self.action_dim))
        
        for iteration in range(self.n_iterations):
            # Sample action sequences from current distribution
            action_sequences = np.random.normal(
                mean[None, :, :],
                std[None, :, :],
                size=(self.population_size, self.horizon, self.action_dim)
            )
            # Clip to action bounds
            action_sequences = np.clip(action_sequences, self.action_low, self.action_high)
            
            # Evaluate each sequence
            returns = self._evaluate_sequences(initial_state, action_sequences)
            
            # Select elite sequences
            elite_indices = np.argsort(returns)[-self.n_elite:]
            elite_sequences = action_sequences[elite_indices]
            
            # Update distribution
            mean = elite_sequences.mean(axis=0)
            std = elite_sequences.std(axis=0) + 1e-6
        
        # Return first action of best sequence
        return mean[0]
    
    def _evaluate_sequences(self, initial_state, action_sequences):
        """Evaluate action sequences using learned model."""
        batch_size = len(action_sequences)
        
        # Convert to torch
        state = torch.FloatTensor(initial_state).unsqueeze(0).repeat(batch_size, 1).to(device)
        
        total_rewards = torch.zeros(batch_size).to(device)
        
        with torch.no_grad():
            for t in range(self.horizon):
                # Get actions for this timestep
                actions = torch.FloatTensor(action_sequences[:, t]).to(device)
                
                # Predict next state and reward
                state, reward = self.dynamics_model.predict(state, actions)
                total_rewards += reward.squeeze()
        
        return total_rewards.cpu().numpy()


# Test planner
planner = CEMPlanner(
    dynamics_model=ensemble,
    action_dim=2,
    horizon=10,
    population_size=200
)
print(f"CEM planner created:")
print(f"  Horizon: {planner.horizon}")
print(f"  Population: {planner.population_size}")
print(f"  Elite size: {planner.n_elite}")

# Test planning
test_state = np.random.randn(4)
planned_action = planner.plan(test_state)
print(f"\nPlanned action shape: {planned_action.shape}")
print(f"Planned action: {planned_action}")

## 5. MBPO-Style Model-Based Policy Optimization

In [ ]:
from collections import deque

class ReplayBuffer:
    """Simple replay buffer."""
    def __init__(self, capacity: int = 100000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(states).to(device),
            torch.FloatTensor(actions).to(device),
            torch.FloatTensor(rewards).unsqueeze(1).to(device),
            torch.FloatTensor(next_states).to(device),
            torch.FloatTensor(dones).unsqueeze(1).to(device)
        )
    
    def __len__(self):
        return len(self.buffer)


class MBPOAgent:
    """Simplified MBPO: Model-Based Policy Optimization.
    
    Combines learned dynamics model with model-free RL (simplified SAC).
    """
    
    def __init__(self, state_dim: int, action_dim: int,
                 rollout_length: int = 5, n_rollouts: int = 400,
                 model_lr: float = 1e-3, policy_lr: float = 3e-4):
        """
        Args:
            rollout_length: Length of model rollouts
            n_rollouts: Number of model rollouts per update
        """
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.rollout_length = rollout_length
        self.n_rollouts = n_rollouts
        
        # Dynamics model (ensemble)
        self.dynamics_model = EnsembleDynamicsModel(
            state_dim, action_dim, n_models=5, lr=model_lr
        )
        
        # Simplified policy (for demonstration)
        self.policy = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim),
            nn.Tanh()
        ).to(device)
        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=policy_lr)
        
        # Buffers
        self.real_buffer = ReplayBuffer(capacity=100000)
        self.model_buffer = ReplayBuffer(capacity=100000)
    
    def select_action(self, state, explore: bool = True):
        """Select action from policy."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action = self.policy(state_tensor).cpu().numpy()[0]
        
        if explore:
            action += 0.1 * np.random.randn(self.action_dim)
            action = np.clip(action, -1, 1)
        
        return action
    
    def train_dynamics_model(self, batch_size: int = 256, n_steps: int = 100):
        """Train dynamics model on real data."""
        if len(self.real_buffer) < batch_size:
            return 0.0
        
        losses = []
        for _ in range(n_steps):
            states, actions, rewards, next_states, _ = self.real_buffer.sample(batch_size)
            loss = self.dynamics_model.train_step(states, actions, next_states, rewards)
            losses.append(loss)
        
        return np.mean(losses)
    
    def generate_model_rollouts(self):
        """Generate synthetic rollouts using learned model."""
        if len(self.real_buffer) < self.n_rollouts:
            return
        
        # Sample start states from real buffer
        start_states = self.real_buffer.sample(self.n_rollouts)[0]
        
        # Generate rollouts
        state = start_states
        for step in range(self.rollout_length):
            with torch.no_grad():
                action = self.policy(state)
                next_state, reward = self.dynamics_model.predict(state, action)
            
            # Store synthetic transitions
            for i in range(len(state)):
                self.model_buffer.push(
                    state[i].cpu().numpy(),
                    action[i].cpu().numpy(),
                    reward[i].cpu().numpy(),
                    next_state[i].cpu().numpy(),
                    False
                )
            
            state = next_state
    
    def train_policy(self, batch_size: int = 256, n_steps: int = 50):
        """Train policy on mixed real + model data (simplified)."""
        if len(self.real_buffer) < batch_size or len(self.model_buffer) < batch_size:
            return 0.0
        
        losses = []
        for _ in range(n_steps):
            # Sample from both buffers
            real_states, _, real_rewards, _, _ = self.real_buffer.sample(batch_size // 2)
            model_states, _, model_rewards, _, _ = self.model_buffer.sample(batch_size // 2)
            
            states = torch.cat([real_states, model_states])
            rewards = torch.cat([real_rewards, model_rewards])
            
            # Simple policy gradient (maximize reward)
            actions = self.policy(states)
            _, pred_rewards = self.dynamics_model.predict(states, actions, use_mean=True)
            
            loss = -pred_rewards.mean()  # Maximize reward
            
            self.policy_optimizer.zero_grad()
            loss.backward()
            self.policy_optimizer.step()
            
            losses.append(loss.item())
        
        return np.mean(losses)


print("MBPO agent created")

## 6. Training and Evaluation on Pendulum

In [ ]:
# Create environment
env = gym.make('Pendulum-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

print(f"Environment: Pendulum-v1")
print(f"State dim: {state_dim}")
print(f"Action dim: {action_dim}")

# Create MBPO agent
mbpo_agent = MBPOAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    rollout_length=3,  # Short rollouts
    n_rollouts=400
)

print("\nMBPO agent initialized")

In [ ]:
def train_mbpo(agent, env, n_episodes: int = 100, warmup: int = 5):
    """Train MBPO agent."""
    episode_rewards = []
    model_losses = []
    
    for episode in tqdm(range(n_episodes)):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            # Select action
            action = agent.select_action(state, explore=True)
            
            # Execute in environment
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store real transition
            agent.real_buffer.push(state, action, reward, next_state, done)
            
            episode_reward += reward
            state = next_state
        
        episode_rewards.append(episode_reward)
        
        # Train after warmup
        if episode >= warmup:
            # Train dynamics model
            model_loss = agent.train_dynamics_model(n_steps=50)
            model_losses.append(model_loss)
            
            # Generate model rollouts
            agent.generate_model_rollouts()
            
            # Train policy
            agent.train_policy(n_steps=50)
        
        if (episode + 1) % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            print(f"Episode {episode+1}/{n_episodes}, Avg Reward: {avg_reward:.2f}")
    
    return episode_rewards, model_losses


# Train MBPO
print("Training MBPO agent...")
mbpo_rewards, mbpo_losses = train_mbpo(mbpo_agent, env, n_episodes=50, warmup=5)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Episode rewards
ax1.plot(mbpo_rewards, alpha=0.6, label='Episode Reward')
window = 5
ax1.plot(np.convolve(mbpo_rewards, np.ones(window)/window, mode='valid'),
         label='Smoothed', linewidth=2)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Return')
ax1.set_title('MBPO Training Progress')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Model loss
ax2.plot(mbpo_losses)
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Model Loss')
ax2.set_title('Dynamics Model Training')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/mbpo_training.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal performance: {np.mean(mbpo_rewards[-10:]):.2f}")

## 7. Visualize Model Predictions

In [ ]:
def visualize_model_accuracy(agent, env, n_steps: int = 100):
    """Compare model predictions with real environment."""
    state, _ = env.reset()
    
    real_states = [state]
    pred_states = [state]
    uncertainties = []
    
    # Rollout
    state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
    for _ in range(n_steps):
        # Real step
        action = agent.select_action(state, explore=False)
        next_state_real, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
        
        # Model prediction
        action_tensor = torch.FloatTensor(action).unsqueeze(0).to(device)
        with torch.no_grad():
            next_state_pred, _ = agent.dynamics_model.predict(
                state_tensor, action_tensor, use_mean=True
            )
            uncertainty = agent.dynamics_model.get_uncertainty(
                state_tensor, action_tensor
            )
        
        real_states.append(next_state_real)
        pred_states.append(next_state_pred.cpu().numpy()[0])
        uncertainties.append(uncertainty.cpu().item())
        
        state = next_state_real
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
    
    real_states = np.array(real_states[:-1])  # Remove last
    pred_states = np.array(pred_states[:-1])
    uncertainties = np.array(uncertainties)
    
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # State dimensions
    for i in range(min(3, state_dim)):
        ax = axes[i//2, i%2] if i < 2 else axes[1, 0]
        ax.plot(real_states[:, i], label='Real', linewidth=2)
        ax.plot(pred_states[:, i], label='Predicted', linewidth=2, linestyle='--')
        ax.set_xlabel('Step')
        ax.set_ylabel(f'State Dim {i}')
        ax.set_title(f'Model Prediction Accuracy (Dim {i})')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Uncertainty
    axes[1, 1].plot(uncertainties)
    axes[1, 1].set_xlabel('Step')
    axes[1, 1].set_ylabel('Uncertainty')
    axes[1, 1].set_title('Model Uncertainty (Ensemble Disagreement)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/tmp/model_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Compute error
    error = np.abs(real_states - pred_states).mean(axis=0)
    print("\nPrediction error per dimension:")
    for i, err in enumerate(error):
        print(f"  Dim {i}: {err:.4f}")
    print(f"Average error: {error.mean():.4f}")
    print(f"Average uncertainty: {uncertainties.mean():.4f}")


# Visualize model accuracy
print("Visualizing model predictions...")
visualize_model_accuracy(mbpo_agent, env, n_steps=100)

## 8. Sample Efficiency Comparison

In [ ]:
print("Sample Efficiency Analysis\n" + "="*50)

# Count samples
real_samples = len(mbpo_agent.real_buffer)
model_samples = len(mbpo_agent.model_buffer)
total_samples = real_samples + model_samples

print(f"Real environment samples: {real_samples}")
print(f"Model-generated samples: {model_samples}")
print(f"Total samples used: {total_samples}")
print(f"\nSample amplification: {total_samples / real_samples:.2f}x")

# Estimate equivalent model-free samples
print(f"\nFor equivalent model-free learning:")
print(f"  Estimated samples needed: ~{total_samples}")
print(f"  MBPO real samples used: {real_samples}")
print(f"  Sample efficiency gain: ~{total_samples / real_samples:.1f}x")

# Visualization
fig, ax = plt.subplots(figsize=(8, 6))
categories = ['Real\nSamples', 'Model\nSamples', 'Total\nSamples']
values = [real_samples, model_samples, total_samples]
colors = ['#3498db', '#e74c3c', '#2ecc71']

bars = ax.bar(categories, values, color=colors, alpha=0.7)
ax.set_ylabel('Number of Samples')
ax.set_title('Sample Usage in MBPO')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, value in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(value)}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/sample_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

### Implementations Completed

1. **Dyna-Q**:
   - Tabular model learning
   - Integrated planning and learning
   - Demonstrated speedup over vanilla Q-learning

2. **Neural Dynamics Models**:
   - Deterministic models (predict next state)
   - Probabilistic models (Gaussian distributions)
   - Ensemble models (uncertainty quantification)

3. **Model Predictive Control (MPC)**:
   - Cross-Entropy Method (CEM) planner
   - Iterative refinement of action sequences
   - Model-based action selection

4. **MBPO-Style Learning**:
   - Short model rollouts from real states
   - Mixed real + synthetic experience
   - Combined model learning and policy optimization

### Key Insights

1. **Sample efficiency**: 2-5× improvement through model-based methods

2. **Ensembles crucial**: Provide uncertainty estimates for robust planning

3. **Short rollouts**: Prevent error accumulation (MBPO key insight)

4. **Planning methods**:
   - Random shooting: Simple but inefficient
   - CEM: Good balance of efficiency and simplicity
   - Gradient-based: Most efficient but requires differentiable models

5. **Model accuracy**: Critical for performance, visualize predictions!

6. **Uncertainty awareness**: Use ensemble disagreement for exploration/safety

### Next Steps

- **Lesson 12**: Exploration strategies (curiosity, count-based)
- Try on more complex environments (MuJoCo)
- Implement latent models (World Models, Dreamer)
- Add prioritized planning (focus on high-uncertainty regions)
- Implement model-based meta-learning

## Exercises

### Implementation Exercises

1. **Implement Dyna-Q+** with exploration bonuses for unvisited states

2. **Add prioritized sweeping** to Dyna-Q and measure speedup

3. **Implement different ensemble sizes** (3, 5, 7, 10) and compare uncertainty estimates

4. **Build gradient-based planner** using backpropagation through model

5. **Implement random shooting** planner and compare with CEM

### Analysis Exercises

6. **Measure model error accumulation** over different rollout lengths

7. **Visualize ensemble predictions** for multiple models on same trajectory

8. **Compare sample efficiency** of MBPO vs pure model-free (SAC)

9. **Analyze effect of rollout length** (1, 3, 5, 10) on MBPO performance

10. **Study uncertainty evolution** as more data is collected

### Application Exercises

11. **Train MBPO on HalfCheetah-v4** and compare with SAC

12. **Implement active learning** (select actions to reduce model uncertainty)

13. **Add pessimistic planning** for safety-critical applications

14. **Implement latent dynamics model** with VAE encoder

15. **Build PETS-style MPC** with full probabilistic ensemble